# Laboratorio #6

* Josue Say - 228801
* Flavio Galán - 22386

## Repositorio

- [Enlace](https://github.com/JosueSay/labs-ds/tree/main/lab6)
- [Modelos](https://drive.google.com/file/d/1QMVn59LvPJGygjhK_w7JFk3fRSCOC3uY/view?usp=sharing)

> *Nota:* se está utilizando python *3.12.3* y para no entrenar nuevamente los modelos se recomienda descargar los modelos por el enlace y descomprimirlos, dejar la carpeta `models` dentro de `lab6`.
>
> En la carpeta `docs/algorithms` se encuentran los documentos de referencias y explicación de los 2 métodos implementados para que se pueda ver un mayor detalle con un workflow base.


In [28]:
# %pip install -r requirements.txt

## Librerias

In [ ]:
# Login y auxiliares
import os
import re
import html
import json
import logging
import unicodedata
from logging.handlers import RotatingFileHandler

# Manejo de texto
import ftfy
from unidecode import unidecode
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

# Manejo de datos
import numpy as np
import pandas as pd

# Visualización
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from tabulate import tabulate

# Hugging Face (solo para tokenizador/encoder y logs)
from transformers import AutoTokenizer, AutoModel, logging as hf_logging
hf_logging.set_verbosity_error()

# Scikit-learn: extracción de características, modelado y métricas
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# Persistencia baseline
from joblib import dump as joblib_dump, load as joblib_load

# PyTorch (para la parte BERT+CNN)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

## Constantes

In [ ]:
# ========== CONFIG GENERAL ==========
os.environ["TOKENIZERS_PARALLELISM"] = "false"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LOGGING_ENABLED   = True
LOG_TO_CONSOLE    = True
LOG_DIR           = "./logs"
LOG_FILE_NAME     = "pipeline.log"
LOG_LEVEL         = logging.INFO
LOG_MAX_BYTES     = 2 * 1024 * 1024       # 2MB
LOG_BACKUP_COUNT  = 3
LOG_SAMPLE_N      = 8

# --------- EJECUCIÓN (on/off) ----------
RUN_FREQUENCIES = True
RUN_NGRAMS      = True
RUN_PLOTS       = True
RUN_MODELS      = True
RUN_THRESHOLD   = False
RUN_BERTCNN     = True

# --------- RUTAS / ARCHIVOS ------------
DATA_DIR     = "data"
RAW_FILE     = "train.csv"
CLEAN_FILE   = "cleaned_train.csv"
USE_CLEANED  = False
DO_CLEANING  = True
SAVE_CLEANED = True

# --------- LIMPIEZA  ---------
REMOVE_URLS       = True
REMOVE_MENTIONS   = True
REMOVE_HASHTAGS   = True
REMOVE_EMOJIS     = True
REMOVE_PUNCT      = True
LOWERCASE         = True
REMOVE_STOPWORDS  = True
STOPWORDS_LANG    = "english"

# --------- BOW / NGRAMS ----------------
VEC_MIN_DF        = 2
NGRAM_MAX         = 2
CHAR_NGRAM_RANGE  = (3,5)

# --------- TRAIN/TEST -------------------
TEST_SIZE         = 0.20
RANDOM_STATE      = 42
CALIBRATE_SVC     = True

# --------- PLOTS ------------------------
SAVE_FIGS         = True
IMAGES_DIR        = "./images"
TOPK_FREQ         = 30
TOPK_NGRAMS       = 20

# --------- BERT + CNN (PyTorch) --------
BERT_MODEL_NAME   = "distilbert-base-uncased"
BERT_MAX_LEN      = 128
BERT_BATCH        = 32
BERT_EPOCHS       = 10
BERT_LR           = 1e-3
CNN_FILTERS       = 32
CNN_KERNEL_SIZE   = 3
CNN_DROPOUT       = 0.5
BERT_TRAINABLE    = False
BERT_THRESHOLD    = 0.50
EARLY_STOP_PATIENCE = 2

# Heurística: ¿mojibake?
MOJIBAKE_PAT = re.compile(
    r"[\uFFFD\u0080-\u009F]"      # U+FFFD o control C1
    r"|Ã.|Â.|â.|Û.|Ò|Ó"
)

# --------- MODELOS (guardar/cargar) ----------
MODELS_DIR              = "./models"

# Baseline (scikit-learn)
SAVE_BASELINE_MODEL     = True
BASELINE_MODEL_FILE     = "baseline_best.joblib"
BASELINE_LOAD_PATH      = os.path.join(MODELS_DIR, BASELINE_MODEL_FILE)

# BERT + CNN (PyTorch + tokenizer)
SAVE_BERT_MODEL         = True
BERT_ARTIFACT_DIR       = "bert_cnn"
BERT_MODEL_SUBDIR       = "model"
BERT_TOKENIZER_SUBDIR   = "tokenizer"

BERT_MODEL_LOAD_PATH    = os.path.join(MODELS_DIR, BERT_ARTIFACT_DIR, BERT_MODEL_SUBDIR)
BERT_TOKENIZER_LOAD_PATH= os.path.join(MODELS_DIR, BERT_ARTIFACT_DIR, BERT_TOKENIZER_SUBDIR)

### Documentación

**Config general y logging:**

| Constante                              | ¿Qué hace?                                            | Tipo / opciones                             | Notas                                                            |
| -------------------------------------- | ----------------------------------------------------- | ------------------------------------------- | ---------------------------------------------------------------- |
| `os.environ["TOKENIZERS_PARALLELISM"]` | Controla el paralelismo de tokenizers HF.             | `"true"` / `"false"`                        | En `"false"` para evitar *warnings* en CPU.               |
| `DEVICE`                               | Dispositivo para PyTorch.                             | `torch.device("cuda")` / `"cpu"`            | Auto-detección por `cuda.is_available()`. |
| `LOGGING_ENABLED`                      | Activa/desactiva logs del pipeline.                   | `bool`                                      | Si `False`, se minimiza salida de logging.                       |
| `LOG_TO_CONSOLE`                       | También imprime logs en consola.                      | `bool`                                      | Útil en desarrollo.                                              |
| `LOG_DIR`                              | Carpeta de logs.                                      | `str` (ruta)                                | Se crea si no existe.                                            |
| `LOG_FILE_NAME`                        | Archivo principal de log.                             | `str`                                       | Rotado con tamaño.                                               |
| `LOG_LEVEL`                            | Nivel de log.                                         | `logging.DEBUG/INFO/WARNING/ERROR/CRITICAL` | Recomendado `INFO` en producción, `DEBUG` al depurar.            |
| `LOG_MAX_BYTES`                        | Tamaño máx. por archivo antes de rotar.               | `int` (bytes)                               | Ej.: `2*1024*1024` (2MB).                                        |
| `LOG_BACKUP_COUNT`                     | Cuántos archivos de respaldo guardar.                 | `int`                                       | Rotación de logs.                                                |
| `LOG_SAMPLE_N`                         | Muestras "antes/después" que se imprimen de limpieza. | `int`                                       | Para inspección rápida.                                          |

**Switches de ejecución (on/off):**

| Constante         | ¿Qué ejecuta?                                      | Tipo / opciones | Notas                                         |
| ----------------- | -------------------------------------------------- | --------------- | --------------------------------------------- |
| `RUN_FREQUENCIES` | Conteo de unigramas y tablas por clase.            | `bool`          | Paso descriptivo.                             |
| `RUN_NGRAMS`      | Top de n-gramas (uni/bi/tri según config).         | `bool`          | Contexto léxico.                              |
| `RUN_PLOTS`       | Nubes de palabras e histogramas.                   | `bool`          | Guarda en `IMAGES_DIR` si `SAVE_FIGS`.        |
| `RUN_MODELS`      | Entrena/evalúa la **línea base** (scikit-learn).   | `bool`          | Genera `baseline_best.joblib` si se guarda.   |
| `RUN_THRESHOLD`   | Búsqueda de umbral óptimo (si el clf tiene proba). | `bool`          | Solo para clasificadores con `predict_proba`. |
| `RUN_BERTCNN`     | Entrena/evalúa **DistilBERT+CNN (PyTorch)**.       | `bool`          | Genera archivos en `models/bert_cnn/`.      |

**Rutas / archivos de datos:**

| Constante      | ¿Qué hace?                             | Tipo / opciones | Notas                                |
| -------------- | -------------------------------------- | --------------- | ------------------------------------ |
| `DATA_DIR`     | Carpeta de datos.                      | `str`           | Ej. `data`.                          |
| `RAW_FILE`     | CSV crudo.                             | `str`           | Ej. `train.csv`.                     |
| `CLEAN_FILE`   | CSV limpio a cachear.                  | `str`           | Ej. `cleaned_train.csv`.             |
| `USE_CLEANED`  | Si existe `CLEAN_FILE`, usarlo. | `bool`          | Ahorra tiempo en corridas repetidas. |
| `DO_CLEANING`  | Limpia el crudo si no hay limpio.      | `bool`          | Aplica tu pipeline de limpieza.      |
| `SAVE_CLEANED` | Guarda el resultado limpio.            | `bool`          | Útil para caché local.               |

**Limpieza:**

| Constante          | ¿Qué hace?                            | Tipo / opciones                      |
| ------------------ | ------------------------------------- | ------------------------------------ |
| `REMOVE_URLS`      | Quita URLs.                           | `bool`                               |
| `REMOVE_MENTIONS`  | Quita `@usuario`.                     | `bool`                               |
| `REMOVE_HASHTAGS`  | Quita `#hashtag` (el token completo). | `bool`                               |
| `REMOVE_EMOJIS`    | Quita emojis y símbolos misceláneos.  | `bool`                               |
| `REMOVE_PUNCT`     | Quita puntuación.                     | `bool`                               |
| `LOWERCASE`        | Convierte a minúsculas.               | `bool`                               |
| `REMOVE_STOPWORDS` | Quita stopwords del idioma.           | `bool`                               |
| `STOPWORDS_LANG`   | Idioma de stopwords.                  | `str` (ej. `"english"`, `"spanish"`) |

**BOW / N-grams:**

| Constante          | ¿Qué hace?                            | Tipo / opciones  | Notas                                    |
| ------------------ | ------------------------------------- | ---------------- | ---------------------------------------- |
| `VEC_MIN_DF`       | Mín. df para incluir un término.      | `int` >= 1        | Sub-muros: elimina términos muy raros. |
| `NGRAM_MAX`        | Máx. n de palabras (1=uni, 2=uni+bi). | `int` >= 1        |                     |
| `CHAR_NGRAM_RANGE` | Rango de n-gramas de **caracteres**.  | `tuple(int,int)` | Ej. `(3,5)` capturan variantes y typos.  |

**Train/Test (scikit-learn):**

| Constante       | ¿Qué hace?                             | Tipo / opciones | Notas                         |
| --------------- | -------------------------------------- | --------------- | ----------------------------- |
| `TEST_SIZE`     | Proporción para test.                  | `float` (0–1)   | Ej. `0.2` = 80/20.            |
| `RANDOM_STATE`  | Semilla de *split*.                    | `int`           | Reproducibilidad.             |
| `CALIBRATE_SVC` | Calibra LinearSVC para probabilidades. | `bool`          | Usa `CalibratedClassifierCV`. |

**Plots:**

| Constante     | ¿Qué hace?                        | Tipo / opciones |
| ------------- | --------------------------------- | --------------- |
| `SAVE_FIGS`   | Guarda figuras a disco.           | `bool`          |
| `IMAGES_DIR`  | Carpeta de imágenes.              | `str`           |
| `TOPK_FREQ`   | Top-K términos para tablas/plots. | `int`           |
| `TOPK_NGRAMS` | Top-K n-gramas (k>=2) para tablas. | `int`           |

**BERT + CNN (PyTorch):**

| Constante             | ¿Qué hace?                            | Tipo / opciones      | Notas                                                   |
| --------------------- | ------------------------------------- | -------------------- | ------------------------------------------------------- |
| `BERT_MODEL_NAME`     | Backbone HF a usar.                   | `str` (ID HF)        |         |
| `BERT_MAX_LEN`        | Tokens máx. del *sequence length*.    | `int` (p.ej. 64–256) | 128 cubre casi todos los tweets.                        |
| `BERT_BATCH`          | Tamaño de batch.                      | `int`                | Ajusta por memoria (CPU/GPU).                           |
| `BERT_EPOCHS`         | Épocas máximas.                       | `int`                |                   |
| `BERT_LR`             | Learning rate de la **cabeza**.       | `float`              |      |
| `CNN_FILTERS`         | Nº de filtros Conv1D.                 | `int`                | 32–128 típico.                                          |
| `CNN_KERNEL_SIZE`     | Ancho del kernel (n-grama "virtual"). | `int`                | 3 = trigrama; puede probarse 3/4/5                      |
| `CNN_DROPOUT`         | Dropout tras pooling.                 | `float` (0–1)        | 0.3–0.5 recomendado.                                    |
| `BERT_TRAINABLE`      | Descongelar/congelar encoder.         | `bool`               | `False` = más rápido en CPU; `True` = mejor si hay GPU. |
| `BERT_THRESHOLD`      | Umbral de decisión para clase 1.      | `float` (0–1)        |                          |
| `EARLY_STOP_PATIENCE` | Paciencia para *early stopping*.      | `int` >= 1            | Evalúa `val_loss`.                                      |

**Heurística de texto ambiguo:**

| Constante      | ¿Qué hace?                      | Tipo / opciones |
| -------------- | ------------------------------- | --------------- |
| `MOJIBAKE_PAT` | Regex para detectar "mojibake". | `re.Pattern`    |

**Modelos (guardar/cargar):**

| Constante                  | ¿Qué hace?                     | Tipo / opciones | Notas                                 |
| -------------------------- | ------------------------------ | --------------- | ------------------------------------- |
| `MODELS_DIR`               | Carpeta raíz de modelos.       | `str`           | Se crea si no existe.                 |
| `SAVE_BASELINE_MODEL`      | Guardar pipeline baseline.     | `bool`          | Guarda vectorizador+clf juntos.       |
| `BASELINE_MODEL_FILE`      | Nombre del .joblib.            | `str`           |            |
| `BASELINE_LOAD_PATH`       | Ruta de carga baseline.        | `str`           |                   |
| `SAVE_BERT_MODEL`          | Guardar artefactos BERT+CNN.   | `bool`          | Guarda tokenizer + `state_dict`/meta. |
| `BERT_ARTIFACT_DIR`        | Subcarpeta de BERT.            | `str`           |                        |
| `BERT_MODEL_SUBDIR`        | Carpeta del **modelo**.        | `str`           |                           |
| `BERT_TOKENIZER_SUBDIR`    | Carpeta del **tokenizer**.     | `str`           |                       |
| `BERT_MODEL_LOAD_PATH`     | Ruta de carga del modelo BERT. | `str`           | `.../bert_cnn/model`.                 |
| `BERT_TOKENIZER_LOAD_PATH` | Ruta de carga del tokenizer.   | `str`           | `.../bert_cnn/tokenizer`.             |

## Helpers

Funciones adicionales que servirán para el código implementado.

In [ ]:
def printTable(df: pd.DataFrame, title: str = "", n: int = 20) -> None:
    if title: print(f"\n=== {title} ===")
    print(tabulate(df.head(n), headers='keys', tablefmt='github', showindex=False))

Se utiliza para imprimir tablas para debug/log.

In [ ]:
def ensureDir(path: str) -> bool:
    try:
        os.makedirs(path, exist_ok=True)
        return True
    except Exception as e:
        print(f"[ensureDir] Error: {e}")
        return False

def initLogger(name: str = "lab6") -> logging.Logger:
    ensureDir(LOG_DIR)
    logger = logging.getLogger(name)
    logger.setLevel(LOG_LEVEL)
    logger.handlers.clear()
    fh = RotatingFileHandler(os.path.join(LOG_DIR, LOG_FILE_NAME),
                             maxBytes=LOG_MAX_BYTES, backupCount=LOG_BACKUP_COUNT,
                             encoding="utf-8")
    fmt = logging.Formatter("[%(asctime)s] %(levelname)s - %(message)s",
                            datefmt="%Y-%m-%d %H:%M:%S")
    fh.setFormatter(fmt); logger.addHandler(fh)
    if LOG_TO_CONSOLE:
        ch = logging.StreamHandler(); ch.setFormatter(fmt); logger.addHandler(ch)
    logger.propagate = False
    if LOGGING_ENABLED:
        logger.info("=== Logger inicializado ===")
    return logger

def logSection(logger: logging.Logger, title: str):
    if LOGGING_ENABLED:
        logger.info("")
        logger.info("=" * (len(title) + 8))
        logger.info(f"==  {title}  ==")
        logger.info("=" * (len(title) + 8))

Este trío forma el "sistema de bitácora" del proyecto:

- `ensureDir` es el guardia de seguridad que se asegura de que exista la carpeta donde escribir (crea `path` con `os.makedirs(..., exist_ok=True)` y devuelve `True` si todo bien o `False` si falló, dejando el error impreso).

- `initLogger` monta sobre esa carpeta un logger.
    - crea (o valida) `LOG_DIR`
    - obtiene un `logging.Logger` con nombre (por defecto `"lab6"`)
    - fija el nivel `LOG_LEVEL`
    - limpia handlers previos para evitar mensajes duplicados
    - añade un `RotatingFileHandler` apuntando a `LOG_FILE_NAME` con rotación por tamaño (`maxBytes=LOG_MAX_BYTES`, `backupCount=LOG_BACKUP_COUNT`, codificación UTF-8) y un formato uniforme `[fecha hora] NIVEL - mensaje`
    - si `LOG_TO_CONSOLE` es `True` agrega también un `StreamHandler` a consola con el mismo formato; desactiva `propagate` para que los mensajes no se repliquen a loggers ancestros y, si `LOGGING_ENABLED` está activo, escribe un **"Logger inicializado"** de bienvenida
    - finalmente devuelve el `logger` configurado.

- `logSection` es un pequeño helper estético que, usando ese mismo `logger`, imprime un salto de línea y un encabezado alrededor del título que se le pase

## Data Cleaning

Hay tres piezas: un detector de "texto roto", un reparador de codificación y el limpiador principal que aplica reglas sobre todo el DataFrame.

**Reparación de codificación (antes de todo):**

* **`looksMojibake(t)`**
  Chequea con una **expresión regular** si el texto trae "símbolos raros" típicos de mala decodificación (`Ã`, `Â`, `�`, etc.).
* **`autoFixEncoding(t)`**

  1. Intenta **arreglar con `ftfy.fix_text`** (arregla comillas curvas, mojibake frecuente, espacios raros).
  2. Si aún "huele a mojibake", prueba una **re-decoding heurística**: toma los bytes como si fueran `latin1/cp1252` y los **decodifica como UTF-8**.
  3. Si nada aplica, **devuelve el original**.
     *¿Para qué?* Para evitar tokens basura tipo "ÂÂfire" que confunden al modelo.

**Limpieza "lingüística" (por columnas):**

La función **`cleaningData(df)`** aplica, en este orden, sobre `df["text"]`:

1. **Relleno de nulos**: `NA -> ""`.
2. **Desescape HTML**: `html.unescape` (convierte `&amp;` → `&`, etc.).
3. **Arreglo de codificación**: `autoFixEncoding` (ver arriba).
4. **Normalización Unicode (NFKC)**: homogeneiza formas equivalentes (espacios, signos, letras compuestas).
5. **Eliminación de URLs** *(si `REMOVE_URLS=True`)*: borra `http(s)://…` y `www.…`.
6. **Eliminación de menciones** *(si `REMOVE_MENTIONS=True`)*: quita `@usuario`.
7. **Eliminación de hashtags** *(si `REMOVE_HASHTAGS=True`)*: quita `#palabra` (el hash y el token).
8. **Eliminación de emojis/símbolos** *(si `REMOVE_EMOJIS=True`)*: limpia rangos U+1F300–1FAFF y ☀–⛿.
9. **Eliminación de puntuación** *(si `REMOVE_PUNCT=True`)*: deja solo letras/dígitos/espacios.
10. **Limpieza de guiones bajos**: colapsa `_` residuales.
11. **Transliteración a ASCII** con **`unidecode`**: `café → cafe`, `niño → nino`. *(En este script está activado por defecto; útil para modelos basados en n-gramas de caracteres.)*
12. **(Opcional) Forzar ASCII-only**: si estuviera activo, eliminaría cualquier carácter no ASCII. *(Aquí está desactivado.)*
13. **Minúsculas** *(si `LOWERCASE=True`)*.
14. **Espacios**: colapsa múltiples espacios y hace *trim*.
15. **Stopwords** *(si `REMOVE_STOPWORDS=True`)*: quita artículos/preposiciones/conjunciones en **inglés** usando `nltk.corpus.stopwords`.

> **Nota de diseño (orden importa):** URLs/mentions/hashtags se quitan **antes** de la puntuación para no romper los patrones; minúsculas se aplican **al final**; la normalización Unicode + `ftfy` van **al inicio** para estabilizar el texto que sigue.

**Finalmente:**

Si `LOGGING_ENABLED=True`, la función:

* Reporta **# de filas** y **nulos en `text`**.
* Muestra **N ejemplos "antes → después"** (controlado por `LOG_SAMPLE_N`).
* Calcula la **longitud media del texto** antes y después y la **% de reducción** (útil para auditar cuánto ruido se quitó).

In [ ]:
def looksMojibake(t: str) -> bool:
    return bool(MOJIBAKE_PAT.search(t))

def autoFixEncoding(t: str) -> str:
    if not t:
        return t
    try:
        fixed = ftfy.fix_text(t)
        if fixed != t:
            return fixed
    except Exception:
        pass
    if looksMojibake(t):
        for enc in ("latin1", "cp1252"):
            try:
                b = t.encode('latin1', errors='ignore')
                return b.decode('utf-8', errors='ignore')
            except Exception:
                pass
    return t

In [ ]:
def cleaningData(df: pd.DataFrame,
                 log: bool = LOGGING_ENABLED,
                 logger: logging.Logger | None = None,
                 sample_n: int = LOG_SAMPLE_N) -> pd.DataFrame:
    if logger is None:
        logger = logging.getLogger("lab6")
    if log:
        logSection(logger, "LIMPIEZA")
        logger.info(f"Filas: {len(df)} | nulos(text): {df['text'].isna().sum()}")

    has_id = 'id' in df.columns
    if log:
        sample = df.sample(min(sample_n, len(df)), random_state=RANDOM_STATE)[['text']].copy()
        if has_id:
            sample = df[['id','text']].loc[sample.index].copy()

    s = df['text'].fillna('').astype(str).apply(html.unescape)
    s = s.apply(autoFixEncoding)
    s = s.apply(lambda t: unicodedata.normalize("NFKC", t))
    if REMOVE_URLS:       s = s.str.replace(r'(https?://\S+|www\.\S+)', ' ', regex=True)
    if REMOVE_MENTIONS:   s = s.str.replace(r'@\w+', ' ', regex=True)
    if REMOVE_HASHTAGS:   s = s.str.replace(r'#\w+', ' ', regex=True)
    if REMOVE_EMOJIS:     s = s.str.replace(r'[\U0001F300-\U0001FAFF\u2600-\u26FF]+', ' ', regex=True)
    if REMOVE_PUNCT:      s = s.str.replace(r"[^\w\s]", " ", regex=True)
    s = s.str.replace(r"_+", " ", regex=True)
    try:
        if True: s = s.apply(unidecode)
    except Exception:
        pass
    s = s.str.replace(r"[^\x00-\x7F]+", " ", regex=True) if False else s
    if LOWERCASE:         s = s.str.lower()
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    if REMOVE_STOPWORDS:
        stp = set(stopwords.words(STOPWORDS_LANG))
        s = s.apply(lambda t: " ".join(w for w in t.split() if w not in stp))

    out = df.copy(); out['text'] = s

    if log:
        lens_before = df['text'].fillna("").astype(str).str.len()
        lens_after  = out['text'].str.len()
        red_mean = (1 - (lens_after.mean() / (lens_before.replace(0,1)).mean())) * 100
        logger.info(f"Longitud media antes: {lens_before.mean():.1f} | después: {lens_after.mean():.1f} | reducción≈{red_mean:.1f}%")
        logger.info("Muestras de limpieza (antes -> después):")
        for idx in sample.index:
            orig = df.loc[idx, 'text']; clean = out.loc[idx, 'text']
            pre = (orig[:180] + "…") if len(orig) > 180 else orig
            pos = (clean[:180] + "…") if len(clean) > 180 else clean
            if has_id: logger.info(f"[id={df.loc[idx,'id']}]")
            logger.info(f"  BEFORE: {pre}"); logger.info(f"  AFTER : {pos}")
    return out

## Análisis previo para implementar modelos

In [ ]:
def loadDataset(dataDir: str = DATA_DIR,
                rawFile: str = RAW_FILE,
                cleanFile: str = CLEAN_FILE,
                useCleaned: bool = True,
                doCleaning: bool = False,
                saveCleaned: bool = False,
                cleaningFn=cleaningData) -> dict:
    clean_path = os.path.join(dataDir, cleanFile)
    raw_path   = os.path.join(dataDir, rawFile)
    artifacts, did_clean = [], False

    if useCleaned and os.path.exists(clean_path):
        df = pd.read_csv(clean_path); used = clean_path
    else:
        df = pd.read_csv(raw_path); used = raw_path
        if doCleaning and (cleaningFn is not None):
            df = cleaningFn(df); did_clean = True
            if saveCleaned:
                ensureDir(dataDir); df.to_csv(clean_path, index=False); artifacts.append(clean_path)

    if 'text' not in df.columns or 'target' not in df.columns:
        raise ValueError("El CSV debe contener 'text' y 'target'.")
    df['text'] = df['text'].fillna('')
    df['target'] = df['target'].astype(int)
    return {'df': df, 'usedFile': used, 'didCleaning': did_clean, 'artifacts': artifacts}

1. **Arma rutas**:

   * `clean_path = dataDir/cleanFile`
   * `raw_path   = dataDir/rawFile`
2. **Decide qué leer**:

   * Si `useCleaned` **y** existe `clean_path` → lee **`clean_path`**.
   * Si no, lee **`raw_path`** (CSV crudo).
3. **Limpia opcionalmente**:

   * Si leyó el crudo **y** `doCleaning=True` **y** hay `cleaningFn`:

     * Aplica `cleaningFn(df)`.
     * Marca `did_clean = True`.
     * Si `saveCleaned=True`, crea carpeta si hace falta y **guarda** `clean_path`, registrándolo en `artifacts`.
4. **Validación de esquema**:

   * Verifica que existan columnas **`text`** y **`target`**. Si faltan → `ValueError`.
5. **Normaliza tipos**:

   * `text`: rellena `NaN` con `""`.
   * `target`: castea a **`int`**.
6. **Devuelve un paquete**:

   ```python
   {
     'df': df,                        # DataFrame listo para modelar
     'usedFile': used,                # ruta del archivo realmente usado (clean o raw)
     'didCleaning': did_clean,        # si se limpió en esta corrida
     'artifacts': artifacts           # rutas de archivos generados (p.ej., cleaned CSV)
   }
   ```

### Frecuencias

In [ ]:
def computeFrequencies(df: pd.DataFrame, minDf: int = VEC_MIN_DF, topK: int = TOPK_FREQ,
                       verbose: bool = True, log: bool = LOGGING_ENABLED, logger=None):
    if logger is None: logger = logging.getLogger("lab6")
    if log: logSection(logger, "FRECUENCIAS (unigramas)")
    cv = CountVectorizer(ngram_range=(1,1), min_df=minDf)
    X = cv.fit_transform(df['text']); vocab = cv.get_feature_names_out()
    if log: logger.info(f"Vocabulario (uni): {len(vocab)} | min_df={minDf}")

    mask_pos = (df['target']==1).values; mask_neg = ~mask_pos
    counts_pos = X[mask_pos].sum(axis=0).A1
    counts_neg = X[mask_neg].sum(axis=0).A1
    counts_all = X.sum(axis=0).A1

    freq_df = pd.DataFrame({'token': vocab,'pos':counts_pos,'neg':counts_neg,'total':counts_all})
    freq_df['lift_pos'] = (freq_df['pos']+1)/(freq_df['neg']+1)
    freq_df['lift_neg'] = (freq_df['neg']+1)/(freq_df['pos']+1)

    top_pos = freq_df.sort_values(['pos','lift_pos'], ascending=False).head(topK)
    top_neg = freq_df.sort_values(['neg','lift_neg'], ascending=False).head(topK)
    intersect = freq_df[(freq_df['pos']>0)&(freq_df['neg']>0)].sort_values('total', ascending=False).head(topK)

    if log:
        logger.info(f"TOP{topK} pos: {', '.join(top_pos['token'].tolist()[:15])}…")
        logger.info(f"TOP{topK} neg: {', '.join(top_neg['token'].tolist()[:15])}…")
        logger.info(f"Intersección TOP{topK}: {', '.join(intersect['token'].tolist()[:15])}…")

    if verbose:
        printTable(top_pos[['token','pos','lift_pos']], "Top en DESASTRES (pos)", n=topK)
        printTable(top_neg[['token','neg','lift_neg']], "Top en NO DESASTRES (neg)", n=topK)
        printTable(intersect[['token','pos','neg','total']], "Términos en ambas categorías", n=topK)

    return {'params': {'minDf': minDf, 'topK': topK},
            'freqDf': freq_df, 'topPos': top_pos, 'topNeg': top_neg,
            'intersect': intersect, 'vectorizer': cv}

Calcula **frecuencias de unigramas** (palabras sueltas) por clase: *desastre* (`target=1`) vs *no desastre* (`target=0`), y arma tablas útiles para explorar vocabulario discriminativo.

1. **Vectoriza** el texto con `CountVectorizer(ngram_range=(1,1), min_df=minDf)`: construye el vocabulario de unigramas y una matriz `X` (documento \* término) con **conteos**.
2. Separa los tweets por clase con máscaras (`mask_pos`, `mask_neg`) y **suma** conteos por término para cada clase: `counts_pos`, `counts_neg`, y totales `counts_all`.
3. Construye `freq_df` con columnas:

    * `token`, `pos` (frecuencia en desastres), `neg` (frecuencia en no desastres), `total`.
    * **Lift** por clase con *suavizado de +1*:

      * `lift_pos = (pos+1)/(neg+1)` (qué tanto un término es más propio de la clase positiva).
      * `lift_neg = (neg+1)/(pos+1)` (análoga para la negativa).
4. Genera tres vistas ordenadas:

    * `top_pos`: términos más **frecuentes** en desastres (desempata con `lift_pos`).
    * `top_neg`: términos más **frecuentes** en no desastres (desempata con `lift_neg`).
    * `intersect`: términos que aparecen en **ambas** clases, ordenados por `total`.
5. Si `log=True`, escribe resúmenes al logger (tamaños y top-terms).
    Si `verbose=True`, imprime tablas bonitas con `printTable(...)`.

### N-Gramas

In [ ]:
def analyzeNgrams(df: pd.DataFrame, nMax: int = NGRAM_MAX, minDf: int = VEC_MIN_DF,
                  topK: int = TOPK_NGRAMS, verbose: bool = True):
    cv = CountVectorizer(ngram_range=(1, nMax), min_df=minDf)
    X = cv.fit_transform(df['text']); vocab = cv.get_feature_names_out()
    mask_pos = (df['target'] == 1).values; mask_neg = (df['target'] == 0).values
    pos_n = X[mask_pos].sum(axis=0).A1; neg_n = X[mask_neg].sum(axis=0).A1
    total = pos_n + neg_n

    df_all = pd.DataFrame({'token': vocab, 'pos': pos_n, 'neg': neg_n, 'total': total})
    df_all['n'] = df_all['token'].str.count(' ') + 1

    byN, tops = {}, {}
    for k in range(2, nMax+1):
        filt = df_all[df_all['n'] == k].sort_values('total', ascending=False)
        tops[k] = filt.head(topK).copy(); byN[k] = filt
        if verbose: printTable(tops[k][['token','pos','neg','total']], f"Top {k}-gramas", n=topK)
    return {'params': {'nMax': nMax, 'minDf': minDf, 'topK': topK}, 'byN': byN, 'tops': tops, 'vectorizer': cv}

Explorar **n-gramas de palabras** (bigramas, trigramas, …) para capturar **contexto** que los unigramas no ven (ej.: `forest fire`, `oil spill`, `heat wave`).


1. **Vectoriza** con `CountVectorizer(ngram_range=(1, nMax), min_df=minDf)`: construye una matriz de conteos que incluye unigramas hasta n-gramas de tamaño `nMax`.
2. Separa por clase (`target==1` vs `target==0`) y **suma** conteos para cada término → `pos_n`, `neg_n`; también calcula el **total** (`pos+neg`).
3. Arma `df_all` con columnas:

    * `token` (el n-grama), `pos`, `neg`, `total`.
    * `n` = longitud del n-grama (contando espacios + 1).
4. Para cada `k` en `2..nMax` filtra los n-gramas de longitud `k`, los **ordena por `total`** (más frecuentes primero) y guarda:

    * `byN[k]`: **todos** los k-gramas con sus conteos.
    * `tops[k]`: el **Top-K** más frecuentes.
      Si `verbose=True`, imprime una tabla bonita de cada Top-K con `printTable(...)`.

### Visualización de data

In [ ]:
def generateWordClouds(topPos: pd.DataFrame, topNeg: pd.DataFrame,
                       saveFigures: bool = SAVE_FIGS, imagesDir: str = IMAGES_DIR, prefix: str = "wc_"):
    artifacts = [];  ensureDir(imagesDir) if saveFigures else None
    # POS
    freq_pos = dict(zip(topPos['token'], topPos['pos']))
    plt.figure(figsize=(8,5))
    wc_pos = WordCloud(width=900, height=400, background_color='white').generate_from_frequencies(freq_pos)
    plt.imshow(wc_pos); plt.axis('off'); plt.title('Desastres (Top Unigramas)')
    if saveFigures:
        path = os.path.join(imagesDir, f"{prefix}desastres.png")
        plt.savefig(path, bbox_inches='tight', dpi=150); artifacts.append(path)
    plt.show()
    # NEG
    freq_neg = dict(zip(topNeg['token'], topNeg['neg']))
    plt.figure(figsize=(8,5))
    wc_neg = WordCloud(width=900, height=400, background_color='white').generate_from_frequencies(freq_neg)
    plt.imshow(wc_neg); plt.axis('off'); plt.title('No desastres (Top Unigramas)')
    if saveFigures:
        path = os.path.join(imagesDir, f"{prefix}no_desastres.png")
        plt.savefig(path, bbox_inches='tight', dpi=150); artifacts.append(path)
    plt.show()
    return {'artifacts': artifacts, 'figures': ['wc_pos','wc_neg']}


In [ ]:
def plotTopHistograms(topPos: pd.DataFrame, topNeg: pd.DataFrame,
                      saveFigures: bool = SAVE_FIGS, imagesDir: str = IMAGES_DIR, prefix: str = "hist_"):
    artifacts = []; ensureDir(imagesDir) if saveFigures else None
    # DESASTRES
    tp = topPos.sort_values('pos', ascending=False).head(20)
    plt.figure(figsize=(10,5))
    plt.bar(tp['token'], tp['pos']); plt.xticks(rotation=70, ha='right')
    plt.title('Top-20 palabras en DESASTRES'); plt.ylabel('Frecuencia'); plt.xlabel('Término')
    plt.tight_layout()
    if saveFigures:
        path = os.path.join(imagesDir, f"{prefix}desastres.png")
        plt.savefig(path, bbox_inches='tight', dpi=150); artifacts.append(path)
    plt.show()
    # NO DESASTRES
    tn = topNeg.sort_values('neg', ascending=False).head(20)
    plt.figure(figsize=(10,5))
    plt.bar(tn['token'], tn['neg']); plt.xticks(rotation=70, ha='right')
    plt.title('Top-20 palabras en NO DESASTRES'); plt.ylabel('Frecuencia'); plt.xlabel('Término')
    plt.tight_layout()
    if saveFigures:
        path = os.path.join(imagesDir, f"{prefix}no_desastres.png")
        plt.savefig(path, bbox_inches='tight', dpi=150); artifacts.append(path)
    plt.show()
    return {'artifacts': artifacts}


## Implementación de modelos

### Primer modelo

In [ ]:
def saveBaselineModel(pipeline, modelsDir: str = MODELS_DIR, filename: str = BASELINE_MODEL_FILE) -> str:
    ensureDir(modelsDir); path = os.path.join(modelsDir, filename); joblib_dump(pipeline, path); return path

def loadBaselineModel(path: str = BASELINE_LOAD_PATH):
    return joblib_load(path)

Se utiliza para guardar y cargar modelos entrenados (primer modelo).